# 方案二：烧结温度 + 气氛热力学校验

## 1. 参数配置


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # fix OpenMP conflict
# ============================================================
# ★★★ 用户配置区 — 修改这里即可 ★★★
# ============================================================

# --- 化学体系 ---
system = ["Y", "Mn", "O"]

# --- 温度扫描 (摄氏度, 内部自动转 K) ---
temperatures_C = [25, 100, 200, 300, 400, 500, 600, 700]

# --- 气氛条件 ---
ATMOSPHERES = {
    "惰性气氛":  {"pO2": 1e-6, "pH2O": 0.0, "pCO2": 0.0},
}

# --- 方案1 凸包初筛阈值 ---
E_ABOVE_HULL_THRESHOLD = 0.2  # eV/atom

# --- Section 9 同气氛多温快照 ---
ATMO_SELECT = "惰性气氛"       # 当前仅配置惰性气氛
T_LIST_C = [25, 300,600]     # 三个温度点 (摄氏度)

# --- Section 6 候选相展示阈值 ---
CANDIDATE_THRESHOLD = 5.0     # eV/atom, 绘图筛选阈值（高值包含所有亚稳相）
MAX_PLOT = 20                 # 绘图最多展示的候选相数量

# --- Section 8 & 9 分解路径分析共享参数 ---
ANALYSIS_TEMPS_C = [25, 100, 250, 400, 550, 700]  # 分解分析温度点 (摄氏度)
E_ABOVE_HULL_DECOMP_THRESH = 0.05       # eV/atom, 分解判定阈值

print(f"System: {'-'.join(system)}")
print(f"Temperature range: {temperatures_C[0]}-{temperatures_C[-1]} C")
print(f"Atmospheres: {len(ATMOSPHERES)} (" + ", ".join(ATMOSPHERES.keys()) + ")")
print(f"Hull threshold: {E_ABOVE_HULL_THRESHOLD} eV/atom")


## 2. 导入库 + API 设置


In [ ]:
import os
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter, PDEntry, GrandPotentialPhaseDiagram
from pymatgen.entries.mixing_scheme import MaterialsProjectDFTMixingScheme
from pymatgen.entries.computed_entries import ComputedStructureEntry
from pymatgen.entries.computed_entries import G_GASES, G_ELEMS
from pymatgen.core.composition import Composition
from pymatgen.core.periodic_table import Element
import matplotlib.pyplot as plt
import numpy as np
from scipy import constants

%matplotlib inline

# 中文字体设置
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# ============================================================
# 加载 API Key (从上级目录 myapi.env)
# ============================================================
env_loaded = load_dotenv(os.path.join("..", "api", "myapi.env"))
API_KEY = os.getenv("MP_API_KEY")
if not API_KEY:
    print("\n错误：未从 .env 文件中读取到 MP_API_KEY。")
    raise ValueError("API Key 配置缺失，程序终止。")
else:
    print(".env 文件加载成功，API Key 已读取。")
    print(f"API Key 前缀：{API_KEY[:8]}...")
    print("所有库导入完成！")


## 3. 获取 MP 数据 (0K 基准)


In [ ]:
# ============================================================
# 优先从方案1导入已处理条目 (含 M3GNet / MACE-MP-0 预测)；如无则从 MP 获取
# ============================================================
import json
import os
from monty.json import MontyDecoder

scheme1_export = os.path.join("..", "方案1", "scheme1_export.json")
scheme1_loaded = False

if os.path.exists(scheme1_export):
    print("📂 检测到方案1导出文件，从本地加载...")
    try:
        with open(scheme1_export, "r", encoding="utf-8") as f:
            export_data = json.load(f, cls=MontyDecoder)

        # 校验体系匹配
        exported_system = export_data.get("system", [])
        if set(exported_system) != set(system):
            print(f"⚠️ 体系不匹配 (导出: {exported_system}, 当前: {system})，改从 MP 获取")
        else:
            entries = export_data["entries"]
            print(f"   从方案1加载 {len(entries)} 条条目")
            print(f"   (MP: {export_data.get("n_entries_mp", "?")}, 模型预测: {export_data.get("n_entries_mace", export_data.get("n_entries_m3gnet", "?"))})")

            # 确保 entry_id 为字符串 (反序列化后偶有类型问题)
            for e in entries:
                if not isinstance(getattr(e, "entry_id", None), str):
                    e.entry_id = str(getattr(e, "entry_id", e.composition.reduced_formula))
            # 给预测条目(M3GNet/MACE-MP-0)补上 parameters.run_type 避免后续警告
            for e in entries:
                if getattr(e, "data", {}).get("source", "") in ("M3GNet", "MACE-MP-0"):
                    if not hasattr(e, "parameters") or e.parameters is None:
                        e.parameters = {}
                    e.parameters.setdefault("run_type", "M3GNet")

            scheme1_loaded = True
    except Exception as e:
        print(f"⚠️ 方案1文件加载失败: {e}，改从 MP 获取")
        import traceback
        traceback.print_exc()

if scheme1_loaded:
    # 方案1 导出的条目已由 MaterialsProjectDFTMixingScheme 修正过
    # 预测条目(M3GNet/MACE-MP-0)不经过该修正（不同势能面），直接与 MP 条目混合构建凸包
    print(f"修正后: {len(entries)} 条计算条目 (来自方案1，已预处理)")

    pd_0k = PhaseDiagram(entries)
    print("0K 凸包相图构建完成 (从方案1导入)")

    plotter_0k = PDPlotter(pd_0k, show_unstable=True)
    plotter_0k.show()
    print()
    print("   -- 上图：0K 凸包上的稳定相 (各温度对比的参照)")
else:
    # 回退：直接从 MP API 获取
    print("🌐 从 Materials Project 数据库获取条目...")
    with MPRester(API_KEY) as mpr:
        entries = mpr.get_entries_in_chemsys(
            system,
            include_structure=True,
            additional_criteria={"thermo_types": ["GGA_GGA+U"]}
        )
        print(f"获取到 {len(entries)} 条计算条目 (0K)")

        scheme = MaterialsProjectDFTMixingScheme()
        entries = scheme.process_entries(entries)
        print(f"修正后: {len(entries)} 条计算条目")

        pd_0k = PhaseDiagram(entries)
        print("0K 凸包相图构建完成")

        plotter_0k = PDPlotter(pd_0k, show_unstable=True)
        plotter_0k.show()
        print()
        print("   -- 上图：0K 凸包上的稳定相 (各温度对比的参照)")


## 4. 引入气相热力学数据


In [ ]:
# ============================================================
# 常见烧结气氛：O2, H2O, CO2
# 数据来源：pymatgen 内置 NIST-JANAF (G_GASES / G_ELEMS)
# O 元素参考态通过 G_ELEMS 获取，分子气体通过 G_GASES 获取
# 单位：eV
# ============================================================

def gas_gibbs_O2(temp_K):
    """O2(g) 标准吉布斯自由能 (eV/分子)
    注：O2 是 O 元素的参考态，通过 G_ELEMS 获取"""
    avail_T = sorted([int(k) for k in G_ELEMS.keys()])
    nearest = min(avail_T, key=lambda x: abs(x - temp_K))
    return 2.0 * G_ELEMS[str(nearest)]['O']  # 2个O原子


def gas_gibbs_H2O(temp_K):
    """H2O(g) 标准吉布斯自由能 (eV/分子)，来自 pymatgen G_GASES"""
    avail_T = sorted([int(k) for k in G_GASES['H2O'].keys()])
    nearest = str(min(avail_T, key=lambda x: abs(x - temp_K)))
    return G_GASES['H2O'][nearest]


def gas_gibbs_CO2(temp_K):
    """CO2(g) 标准吉布斯自由能 (eV/分子)，来自 pymatgen G_GASES"""
    avail_T = sorted([int(k) for k in G_GASES['CO2'].keys()])
    nearest = str(min(avail_T, key=lambda x: abs(x - temp_K)))
    return G_GASES['CO2'][nearest]



print("气相吉布斯自由能函数定义完成 (数据来源: pymatgen G_GASES/G_ELEMS)")
def get_element_gibbs(element, temp_K):
    """
    查询元素在温度 T 下的吉布斯自由能 G(T) (eV/atom)
    数据来源: pymatgen G_ELEMS / G_GASES (NIST-JANAF)
    自动选取该温度下最稳定相 (固/液/气)
    """
    avail_T = sorted([int(k) for k in G_ELEMS.keys()])
    nearest = str(min(avail_T, key=lambda x: abs(x - temp_K)))
    return G_ELEMS[nearest][element]

print(f"   O2(298K)  : {gas_gibbs_O2(298):.4f} eV/mol  (G_ELEMS)")
print(f"   H2O(298K) : {gas_gibbs_H2O(298):.4f} eV/mol  (G_GASES)")
print(f"   CO2(298K) : {gas_gibbs_CO2(298):.4f} eV/mol  (G_GASES)")


## 5. 核心函数：GibbsEntrySet + 气氛依赖相图


In [ ]:
def compute_G_delta(entry, T):
    """
    计算 G^delta(T) (eV/atom) — Bartel et al., Nat. Commun. 2018, Eq.4
    基于 SISSO 描述符：仅需 DFT 晶胞体积 + 元素组成，无需声子谱
    适用温度范围: 300–1800 K，精度 ~50 meV/atom (与 QHA 相当)
    """
    import numpy as np
    
    # 原子体积 V = 晶胞体积 / 原子数  (Ang^3/atom)
    structure = entry.structure
    V = structure.volume / structure.num_sites
    
    # 约化原子质量 m_red (加权调和平均, amu)
    comp = entry.composition
    elements = comp.elements
    alpha = [comp[el] for el in elements]
    m = [el.atomic_mass for el in elements]
    total = sum(alpha)
    m_red = total / sum(a / mi for a, mi in zip(alpha, m))
    
    # SISSO 描述符 (Eq.4)
    G_delta = (-2.48e-4 * np.log(V) - 8.94e-5 / m_red) * T
    return G_delta  # eV/atom

def build_gibbs_entries(entries_0k, temperature_K, pO2_atm=0.21):
    """
    构建 Gibbs 条目集 (GibbsEntrySet 概念等价实现)
    对每个 DFT 条目施加完整的高温自由能修正：
      ΔG_f(T) = ΔH_f(298K) + n·G^δ_SISSO(T) − Σ(α_i·G_i(T))
    其中 G_i(T) 取自 pymatgen 内置 NIST-JANAF 热力学表
    O 元素额外包含气氛修正: μ_O = G_O(T) + 1/2·k_B·T·ln(pO2/p0)
    """
    k_B = constants.k / constants.e
    p0 = 1.0

    # 计算 O 化学势（含气氛修正）
    mu_O2 = gas_gibbs_O2(temperature_K)
    mu_O2 += k_B * temperature_K * np.log(pO2_atm / p0)
    mu_O = mu_O2 / 2.0

    gibbs_entries = []
    for e in entries_0k:
        comp = e.composition
        n_atoms = comp.num_atoms

        # SISSO 高温振动修正
        delta_G_vib = n_atoms * compute_G_delta(e, temperature_K)

        # Bartel Eq.2: 完整元素 Gibbs 扣除
        gp_total = e.energy + delta_G_vib
        for el, amt in comp.items():
            if el.symbol == "O":
                gp_total -= amt * mu_O  # O 含气氛修正
            else:
                gp_total -= amt * get_element_gibbs(el.symbol, temperature_K)

        gibbs_entries.append(PDEntry(comp, gp_total))

    return gibbs_entries, mu_O

def build_atmosphere_pd(entries_0k, temperature_K, pO2_atm=0.21):
    """构建气氛依赖凸包相图（调用 GibbsEntrySet → PhaseDiagram）"""
    gibbs_entries, mu_O = build_gibbs_entries(entries_0k, temperature_K, pO2_atm)
    pd = PhaseDiagram(gibbs_entries)
    return pd, mu_O, gibbs_entries

# 验证函数
print("GibbsEntrySet + build_atmosphere_pd 定义完成")
_demo_pd, _demo_mu, _ = build_atmosphere_pd(entries, 900, 0.21)
print(f"  示例: air@900K, mu_O = {_demo_mu:.4f} eV/atom")
_demo_pd2, _demo_mu2, _ = build_atmosphere_pd(entries, 900, 1e-6)
print(f"  示例: Ar@900K,  mu_O = {_demo_mu2:.4f} eV/atom")


## 6. 多温度对比：绘制 T-e_above_hull 曲线


In [ ]:
# 筛选含所有目标元素的化合物
# 不做任何过滤：方案1的全部条目都参与温度校验和导出
compounds = list(entries)

# 按 reduced_formula 去重，保留能量最低的 polymorph
best = {}
for e in compounds:
    rf = e.composition.reduced_formula
    h = pd_0k.get_e_above_hull(PDEntry(e.composition, e.energy))
    if rf not in best or (e.energy_per_atom < best[rf].energy_per_atom):
        best[rf] = e
compounds = list(best.values())

print(f"方案1输入条目: {len(entries)} 个（单质+二元+三元+亚稳态，全部参与）")
print(f"去重后: {len(compounds)} 个独立化学式")
for e in sorted(compounds, key=lambda x: x.composition.reduced_formula)[:10]:
    pdentry = PDEntry(e.composition, e.energy)
    eh = pd_0k.get_e_above_hull(pdentry)
    src = getattr(e.data, "source", "?") if hasattr(e, "data") else "?"
    print(f"   - {e.composition.reduced_formula:20s}  e_hull={eh:.4f} eV/atom  source={src}")


In [ ]:
# ============================================================
# 温度-气氛联合扫描
# ============================================================
stability_data = {}
for atmo_name, atmo_params in [(ATMO_SELECT, ATMOSPHERES[ATMO_SELECT])]:
    atmo_data = {}
    for temp_C in temperatures_C:
        temp = temp_C + 273.15  # C -> K
        gibbs_entries, mu_O = build_gibbs_entries(entries, temp, atmo_params["pO2"])
        pd_atmo = PhaseDiagram(gibbs_entries)
        gibbs_map = {ge.composition.reduced_formula: ge.energy / ge.composition.num_atoms
                     for ge in gibbs_entries}
        for e in compounds:
            comp = e.composition
            n_atoms = comp.num_atoms
            gp_per_atom = gibbs_map.get(comp.reduced_formula)
            if gp_per_atom is None:
                e_above = None
            else:
                try:
                    e_above = gp_per_atom - pd_atmo.get_hull_energy(comp) / n_atoms
                except Exception:
                    e_above = None
            eid = getattr(e, "entry_id", "N/A")
            if eid not in atmo_data:
                atmo_data[eid] = {"formula": comp.formula, "e_above": {}}
            atmo_data[eid]["e_above"][temp_C] = e_above  # store with Celsius key
    stability_data[atmo_name] = atmo_data

print(f"Scanned: {len(temperatures_C)} T x {len(ATMOSPHERES)} atm")
for atmo_name in [ATMO_SELECT]:
    print(chr(10) + "--- " + atmo_name + " ---")
    for temp_C in temperatures_C:
        ad = stability_data[atmo_name]
        stable = [eid for eid, d in ad.items()
                  if d["e_above"].get(temp_C) is not None and d["e_above"][temp_C] < 0.001]
        ranked = sorted([(eid, d["e_above"].get(temp_C)) for eid, d in ad.items()
                        if d["e_above"].get(temp_C) is not None and d["e_above"][temp_C] >= 0.001],
                       key=lambda x: x[1])[:5]
        if stable:
            print(f"  {temp_C}C stable: {stable}")
        if ranked:
            r_str = ", ".join(f"{e}={int(v*1000)}meV" for e, v in ranked)
            print(f"  {temp_C}C near:  {r_str}")


## 7. 可视化：温度-气氛稳定性曲线


In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'

# ============================================================
# 筛选接近稳定的候选化合物
# ============================================================
threshold = CANDIDATE_THRESHOLD  # eV/atom (包含所有三元亚稳相，观察温度趋势)

candidates = []
for e in compounds:
    pdentry = PDEntry(e.composition, e.energy)
    e_above_0k = pd_0k.get_e_above_hull(pdentry)
    if e_above_0k is not None and e_above_0k < threshold:
        candidates.append(e)

print(f"0K 下 e_above_hull < {threshold} eV/atom 的候选相：{len(candidates)} 个")
for e in candidates:
    pdentry = PDEntry(e.composition, e.energy)
    print(f"   - {e.composition.reduced_formula}  ({e.entry_id})  ->  {pd_0k.get_e_above_hull(pdentry):.4f} eV/atom")


In [ ]:
# ============================================================
# 画 T - e_above_hull 曲线 (三种气氛并排对比)
# 只展示最接近凸包的前 20 个候选相，避免图例溢出
# ============================================================

# 按 0K e_above_hull 排序，取前 MAX_PLOT 个
candidates_sorted = sorted(
    candidates,
    key=lambda e: pd_0k.get_e_above_hull(PDEntry(e.composition, e.energy))
)[:MAX_PLOT]
print(f"绘图展示前 {len(candidates_sorted)} 个候选相 (共 {len(candidates)} 个)")

fig, axes = plt.subplots(1, 1, figsize=(6, 5),
                         squeeze=False)

for ax_idx, (atmo_name, atmo_params) in enumerate([(ATMO_SELECT, ATMOSPHERES[ATMO_SELECT])]):
    ax = axes[0, ax_idx]
    atmo_data = stability_data[atmo_name]

    for e in candidates_sorted:
        eid = getattr(e, 'entry_id', 'N/A')
        if eid in atmo_data:
            T_vals = []
            hull_vals = []
            for T in temperatures_C:
                v = atmo_data[eid]["e_above"].get(T)
                if v is not None:
                    T_vals.append(T)
                    hull_vals.append(v * 1000)  # 转为 meV/atom
            if len(T_vals) > 0:
                label = e.composition.reduced_formula
                ax.plot(T_vals, hull_vals, "-o", markersize=3, label=label, linewidth=1.0)

    ax.axhline(y=0, color="red", linestyle="--", linewidth=0.8, label="凸包线 (0 meV)")
    ax.set_xlabel("温度 (C)", fontsize=11)
    ax.set_ylabel("e_above_hull (meV/atom)", fontsize=11)
    ax.set_title(f"{atmo_name}\nO2={atmo_params["pO2"]} atm", fontsize=12)
    ax.legend(fontsize=6, loc="upper left", ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n   -- 曲线越趋近 0 meV (红线)，该相在该温度/气氛下越稳定")
print("   -- 曲线向上走 -> 高温下变得更不稳定 (可能分解)")
print("   -- 曲线向下走 -> 高温下可能稳定化")


## 8. 同气氛多温度凸包快照


In [ ]:
# ============================================================
# 同一气氛下三种温度的凸包对比
# ★ 修改下面两行即可切换气氛和温度
# ============================================================

import plotly.io as pio

atmo_params = ATMOSPHERES[ATMO_SELECT]
for temp_C in T_LIST_C:
    temp = temp_C + 273.15  # C -> K
    pd_atmo, mu_O, gibbs_entries = build_atmosphere_pd(entries, temp, atmo_params["pO2"])

    stable_phases = sorted(set(
        e.composition.reduced_formula
        for e in gibbs_entries
        if pd_atmo.get_e_above_hull(e) is not None
        and pd_atmo.get_e_above_hull(e) < 0.001
    ))
    phase_str = ", ".join(stable_phases) if stable_phases else "(none)"
    print()
    print("=" * 50)
    print(ATMO_SELECT, " T=", temp, "K", " pO2=", atmo_params["pO2"], " atm", sep="")
    print("=" * 50)
    print(f"  mu(O) = {mu_O:.4f} eV/atom")
    print(f"  Stable ({len(stable_phases)}): {phase_str}")
    print()

    plotter = PDPlotter(pd_atmo, show_unstable=True)
    plotter.show()

print()
print("Same atmosphere, three temperatures_C — watch for phase stability changes")


## 9. 汇总与输出


In [ ]:
# ============================================================
# 输出 temperature_analysis.md
# 汇总各候选相在不同气氛下的热稳定性变化
# ============================================================

system_name = "-".join(system)
md_lines = []
md_lines.append(f"# {system_name} 体系 烧结温度 + 气氛热力学校验\n")
md_lines.append(f"\n**温度扫描范围**: {temperatures_C[0]}-{temperatures_C[-1]} C\n")
md_lines.append(f"\n**气氛条件**: {ATMO_SELECT}\n")
md_lines.append("\n---\n")
md_lines.append("\n## 候选相稳定性趋势\n")
md_lines.append("\n| 化学式 | entry_id | 0K e_above (eV) |")
for atmo_name in [ATMO_SELECT]:
    md_lines.append(f" {atmo_name} 趋势 |")
md_lines.append("\n|---|----------|-----------------|")
for _ in [ATMO_SELECT]:
    md_lines.append("---|")
md_lines.append("\n")

for e in candidates:
    pdentry = PDEntry(e.composition, e.energy)
    e0k = pd_0k.get_e_above_hull(pdentry)
    md_lines.append(f"| {e.composition.reduced_formula} | {e.entry_id} | {e0k:.4f} |")

    for atmo_name in [ATMO_SELECT]:
        atmo_data = stability_data[atmo_name]
        if e.entry_id in atmo_data:
            vals = [v for v in atmo_data[e.entry_id]["e_above"].values() if v is not None]
            if len(vals) >= 2:
                trend = vals[-1] - vals[0]  # 高温 - 低温
                if trend < -0.01:
                    trend_str = f"稳定化 ({trend*1000:+.0f} meV)"
                elif trend > 0.01:
                    trend_str = f"不稳定化 ({trend*1000:+.0f} meV)"
                else:
                    trend_str = "基本不变"
            else:
                trend_str = "--"
        md_lines.append(f" {trend_str} |")
    md_lines.append("\n")

md_lines.append("\n---\n")
md_lines.append("\n## 高温分解路径（代表性温度点）\n")
md_lines.append("\n| 化学式 | 气氛 | T (K) | 分解反应 | e_above (meV/atom) |\n")
md_lines.append("|---|---|---|----------|----|\n")
# 汇总分解路径
for atmo_name in [ATMO_SELECT]:
    for temp_C in ANALYSIS_TEMPS_C:
        temp = temp_C + 273.15  # C -> K
        gibbs_entries_rpt, mu_rpt = build_gibbs_entries(entries, temp, ATMOSPHERES[atmo_name]["pO2"])
        pd_rpt = PhaseDiagram(gibbs_entries_rpt)
        for e in candidates:
            comp = e.composition
            n_atoms = comp.num_atoms
            # 从 GibbsEntrySet 查找已修正能量
            gp_total_rpt = None
            for ge in gibbs_entries_rpt:
                if ge.composition.reduced_formula == comp.reduced_formula:
                    gp_total_rpt = ge.energy
                    break
            if gp_total_rpt is None:
                continue
            try:
                hull_rpt = pd_rpt.get_hull_energy(comp) / n_atoms
                e_above_rpt = gp_total_rpt / n_atoms - hull_rpt
            except Exception:
                continue
            if e_above_rpt is not None and e_above_rpt > E_ABOVE_HULL_DECOMP_THRESH:
                try:
                    decomp_rpt = pd_rpt.get_decomposition(comp)
                    if len(decomp_rpt) > 1:
                        prods = [f"{amt:.2f}{pd_e.composition.reduced_formula}" for pd_e, amt in decomp_rpt.items()]
                        rxn = " → ".join([comp.reduced_formula, " + ".join(prods)])
                        md_lines.append(f"| {comp.reduced_formula} | {atmo_name} | {temp} | {rxn} | {e_above_rpt*1000:.0f} |\n")
                except Exception:
                    pass
with open("temperature_analysis.md", "w", encoding="utf-8") as f:
    f.writelines(md_lines)

print("温度气氛分析结果已导出至 temperature_analysis.md")

print(f"\n{'='*70}")
print(f"汇总：{system_name} 体系烧结校验")
print(f"{'='*70}")
print(f"\n共 {len(candidates)} 个候选相在 0K 下 e_above_hull < {threshold} eV/atom")
print(f"扫描 {len(temperatures_C)} 个温度 x {1} 种气氛")
print(f"\n建议下一步：")
print(f"   1. 检查 output.md (方案一) 中标记为亚稳态的相")
print(f"   2. 对照本分析中这些相在高温下的稳定性趋势")
print(f"   3. 优先选择在各气氛下均保持稳定的相作为实验候选")


## 10. 高温分解反应路径


In [ ]:
# ============================================================
# High-temperature decomposition pathway analysis (rxn-network)
# Output appended to temperature_analysis.md
# ============================================================

from rxn_network.entries.entry_set import GibbsEntrySet
from rxn_network.enumerators.basic import BasicEnumerator
from rxn_network.network.network import ReactionNetwork
from rxn_network.costs.functions import Softplus

NL = chr(10)  # newline shortcut
out_file = open("temperature_analysis.md", "a", encoding="utf-8")
out_file.write(NL + "## 高温分解反应路径" + NL + NL)

print("=" * 70)
print("High-T decomposition pathways (rxn-network multi-step)")
print("=" * 70)

analysis_temps_C = ANALYSIS_TEMPS_C
E_ABOVE_THRESH = E_ABOVE_HULL_DECOMP_THRESH

for atmo_name, atmo_params in [(ATMO_SELECT, ATMOSPHERES[ATMO_SELECT])]:
    print()
    print("=" * 60)
    print(f"  {atmo_name}  (pO2 = {atmo_params["pO2"]} atm)")
    print("=" * 60)
    out_file.write(f"### {atmo_name} (pO2={atmo_params["pO2"]} atm)" + NL + NL)

    for temp_C in analysis_temps_C:
        T = temp_C + 273.15
        gibbs_entries, mu_O = build_gibbs_entries(entries, T, atmo_params["pO2"])
        pd_atmo = PhaseDiagram(gibbs_entries)

        print()
        print(f"  T = {temp_C} C  (mu_O = {mu_O:.4f} eV/atom)")
        out_file.write(f"**T = {temp_C} C** (mu_O = {mu_O:.4f} eV/atom)" + NL + NL)

        decomposing = []
        for e in candidates:
            comp = e.composition
            for ge in gibbs_entries:
                if ge.composition.reduced_formula == comp.reduced_formula:
                    gp = ge.energy / comp.num_atoms
                    try:
                        e_above = gp - pd_atmo.get_hull_energy(comp) / comp.num_atoms
                        if e_above is not None and e_above > E_ABOVE_THRESH:
                            decomposing.append((e, e_above, ge))
                    except Exception:
                        import traceback; traceback.print_exc()
                        pass
                    break

        if not decomposing:
            print("    (all candidates stable)")
            out_file.write("All candidates stable at this temperature." + NL + NL)
            continue

        print(f"    {len(decomposing)} metastable candidates")

        try:
            from pymatgen.entries.computed_entries import ComputedEntry
            _ces = [ComputedEntry(ge.composition, ge.energy) for ge in gibbs_entries]
            entry_set = GibbsEntrySet.from_computed_entries(_ces, T)
            be = BasicEnumerator()
            rxns = be.enumerate(entry_set)
            cf = Softplus(temp=T)
            network = ReactionNetwork(rxns, cf)
            network.build()
            _has_network = True
        except Exception:
            _has_network = False

        for e, e_above, ge in decomposing:
            rf = e.composition.reduced_formula
            try:
                decomp = pd_atmo.get_decomposition(e.composition)
                targets = [pd_e.composition.reduced_formula for pd_e in decomp.keys()]
            except Exception:
                continue
            if len(targets) <= 1:
                continue

            if _has_network:
                network.set_precursors([rf])
                network.set_target(targets[0])
                paths = network.find_pathways(targets, k=3)

            # One-step decomposition reaction
            prods = [f"{amt:.2f}{pd_e.composition.reduced_formula}" for pd_e, amt in decomp.items()]
            one_step = " -> ".join([rf] + prods)

            msg = f"    -- {rf} (e_above={e_above*1000:.0f} meV/atom) --"
            print(msg)
            print(f"      One-step: {one_step}")
            out_file.write("- **" + rf + "** (e_above=" + str(int(e_above*1000)) + " meV/atom)" + NL)
            out_file.write("  One-step: " + one_step + NL)
            if _has_network and len(paths.paths) > 0:
                best = paths.paths[0]
                total_cost = sum(best.costs)
                print(f"      Best pathway (cost={total_cost:.3f}):")
                out_file.write(f"  Best pathway (cost={total_cost:.3f}):" + NL)
                for rxn, cost in zip(best.reactions, best.costs):
                    line = f"        {rxn}  (dG={rxn.energy_per_atom:+.3f} eV/atom)"
                    print(line)
                    out_file.write(f"    - {rxn}  (dG={rxn.energy_per_atom:+.3f} eV/atom)" + NL)
            else:
                out_file.write("  No multi-step path available." + NL)
            out_file.write(NL)

out_file.write("---" + NL)
out_file.close()
print()
print("Decomposition pathways appended to temperature_analysis.md")



## 11. 导出 Gibbs 修正条目（供方案3_v2 使用）

对方案1的全部条目施加 Bartel SISSO 有限温度 Gibbs 修正，
导出为 。方案3_v2 的反应网络将基于这些
温度修正后的能量值计算反应自由能和 Softplus 成本。


In [ ]:
# ============================================================
# 导出 Gibbs 修正条目 -> scheme2_gibbs_export.json
# 每个条目导出绝对总能（与方案1一致），不缩放到最简式
# ============================================================
import json as _json
from datetime import datetime as _dt
from monty.json import MontyEncoder as _ME
from pymatgen.entries.computed_entries import ComputedStructureEntry as _CSE
from pymatgen.analysis.phase_diagram import PDEntry as _PDE

T_SYNTH = 900.0
P_O2 = ATMOSPHERES[ATMO_SELECT]["pO2"]

print(f"Gibbs 修正 @ T={T_SYNTH} K, pO2={P_O2} atm")
print(f"体系: {system}, 原始条目: {len(entries)}")

# ---- polymorph 去重（用形成能排序，导出仍用绝对总能）----
# 只做 polymorph 去重：同一最简式保留形成能最低的条目
best_idx = {}  # reduced_formula -> (index, form_energy_per_atom)
for i, e in enumerate(entries):
    comp = e.composition
    if len(comp.elements) == 1:
        form_energy = 0.0  # 单质形成能 = 0
    else:
        n_atoms = comp.num_atoms
        pde = _PDE(comp, e.energy)
        form_0k = pd_0k.get_form_energy(pde)
        if form_0k is None:
            continue
        form_energy = form_0k + n_atoms * compute_G_delta(e, T_SYNTH)  # per cell
    rf = comp.reduced_formula
    form_pa = form_energy / comp.num_atoms if form_energy != 0 else 0
    if rf not in best_idx or form_pa < best_idx[rf][1]:
        best_idx[rf] = (i, form_pa)

# ---- 构建 ComputedStructureEntry (每条目独立晶胞能量) ----
def _fix_keys(obj):
    if isinstance(obj, dict): return {str(k): _fix_keys(v) for k, v in obj.items()}
    if isinstance(obj, list): return [_fix_keys(v) for v in obj]
    return obj

gibbs_entries = []
bad = 0
for i, e in enumerate(entries):
    rf = e.composition.reduced_formula
    if rf not in best_idx:
        continue
    best_i, _ = best_idx[rf]
    if i != best_i:
        continue  # 只保留最低形成能的 polymorph
    comp = e.composition
    # 绝对总能表示（与方案1一致）：0K 绝对总能 + 高温振动修正
    g_total = e.energy + comp.num_atoms * compute_G_delta(e, T_SYNTH)
    try:
        ne = _CSE(structure=e.structure, energy=g_total,
                  entry_id=str(getattr(e, "entry_id", rf)).replace("{","").replace("}","").replace(":",""),
                  correction=getattr(e, "correction", 0.0),
                  data=_fix_keys(dict(getattr(e, "data", {}))))
        ne.data["gibbs_corrected"] = True; ne.data["gibbs_T"] = T_SYNTH; ne.data["gibbs_absolute_energy"] = True
        gibbs_entries.append(ne)
    except Exception:
        bad += 1
        try: fd = _json.loads(_json.dumps(_fix_keys(getattr(e, "data", {})), cls=_ME))
        except: fd = {}
        ne = _CSE(structure=e.structure, energy=g_total,
                  entry_id=str(getattr(e, "entry_id", rf)), correction=0.0, data=fd)
        ne.data["gibbs_corrected"] = True; ne.data["gibbs_T"] = T_SYNTH; ne.data["gibbs_absolute_energy"] = True
        gibbs_entries.append(ne)

if bad: print(f"{bad} entries with bad data keys, fixed")

print(f"修正后: {len(gibbs_entries)} entries (晶胞级绝对总能, polymorph去重)")
for e in gibbs_entries[:5]:
    rf = e.composition.reduced_formula
    isel = "ELEM" if len(e.composition.elements) == 1 else "CMPD"
    n = e.composition.num_atoms
    print(f"  [{isel}] {rf:20s} G_abs={e.energy:+.1f} eV/cell  ({e.energy/n:+.3f} eV/atom)")

# ---- Export ----
export_data = {
    "schema_version": 3, "created_at": _dt.now().isoformat(),
    "system": system, "energy_type": "absolute_total", "gibbs_temperature_K": T_SYNTH, "pO2_atm": P_O2,
    "n_entries_total": len(gibbs_entries),
    "entries": [e.as_dict() for e in gibbs_entries],
}
with open("scheme2_gibbs_export.json", "w", encoding="utf-8") as f:
    _json.dump(export_data, f, cls=_ME, indent=2)
print(f"已导出 ({os.path.getsize('scheme2_gibbs_export.json')/1048576:.1f} MB)")
